# SFT Distillation: Training Qwen3-4B on Claude Reasoning Traces (Unsloth)

**Author:** Behrooz Azarkhalili

| Feature | Detail |
|---------|--------|
| Framework | Unsloth (FastModel) |
| Model | `unsloth/Qwen3-4B` |
| Alternative | unsloth/Qwen3.5-2B |
| Dataset | Claude reasoning distillation traces |
| Method | LoRA (QLoRA 4-bit) |
| Params | ~4B |
| VRAM | 5 GB min |
| GGUF Export | Built-in via `save_pretrained_gguf()` |

In [ ]:
# ============================================================================
# Install (uncomment for Colab / bare-metal)
# ============================================================================
# !pip install unsloth datasets tqdm

# SLURM (Fir cluster):
#   module load gcc arrow python/3.11.5
#   source /scratch/$USER/venvs/hf_unsloth/bin/activate

In [ ]:
from __future__ import annotations
import json, os, torch

SMOKE_TEST: bool = False  # Set False for full training

MODEL_NAME = "unsloth/Qwen3-4B"
# MODEL_NAME = "unsloth/Qwen3.5-2B"  # Alternative
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
HUB_MODEL_ID = "ermiaazarkhalili/Qwen3-4B-SFT-Claude-Opus-Reasoning-Unsloth"
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
MAX_STEPS = 5 if SMOKE_TEST else -1
NUM_EPOCHS = 1 if SMOKE_TEST else 1
BATCH_SIZE = 2
GRAD_ACCUM = 4

print(f"[OK] Model: {MODEL_NAME}")
print(f"[OK] SMOKE_TEST: {SMOKE_TEST}, max_steps: {MAX_STEPS}")
print(f"[OK] PyTorch: {torch.__version__}")
print(f"[OK] GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================================
# HF Authentication
# ============================================================================
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError: pass
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception: pass
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("[OK] Authenticated with HF Hub")
else:
    print("[WARN] No HF_TOKEN found")

In [ ]:
# ============================================================================
# Load Model with Unsloth
# ============================================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"[OK] Model loaded: {MODEL_NAME}")

In [ ]:
# ============================================================================
# Apply LoRA
# ============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none", random_state=3407,
)
print(f"[OK] LoRA applied: r={LORA_R}, alpha={LORA_ALPHA}")

In [ ]:
# ============================================================================
# Chat Template (Qwen3.5 uses built-in template — no get_chat_template needed)
# ============================================================================
print(f"[OK] Using built-in Qwen3.5 chat template")

In [ ]:
# ============================================================================
# Load & Process Claude Reasoning Distillation Dataset
# ============================================================================
from datasets import load_dataset

dataset = load_dataset("ermiaazarkhalili/claude-reasoning-distillation", "sft", split="train")
print(f"[OK] Loaded {len(dataset):,} samples")
if SMOKE_TEST:
    dataset = dataset.select(range(100))
    print(f"[SMOKE] Truncated to {len(dataset)} samples")

def format_distillation(sample):
    messages = []
    for msg in sample["messages"]:
        m = dict(msg)
        m.pop("thinking", None)  # Drop thinking field — model learns from content
        messages.append(m)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_distillation)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
print(f"[OK] Processed: {len(dataset):,} samples")
print(dataset[0]["text"][:200])

In [ ]:
# ============================================================================
# Training
# ============================================================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset, eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5, max_steps=MAX_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE, logging_steps=1,
        optim="adamw_8bit", weight_decay=0.001,
        lr_scheduler_type="linear", seed=3407,
        output_dir="outputs", report_to="none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
start_mem = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"[OK] GPU: {gpu_stats.name}, {max_memory} GB, reserved: {start_mem} GB")
print(f"[....] Training (max_steps={MAX_STEPS}, num_epochs={NUM_EPOCHS}) ...")

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n[OK] Training complete!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
print(f"  Peak VRAM: {used} GB ({round(used/max_memory*100, 1)}%)")

In [ ]:
# ============================================================================
# Inference Test
# ============================================================================
test_prompts = [
    "Check if the numbers 8 and 1233 are powers of two.",
    "What's the weather like in New York today?",
    "Calculate the average of: 10, 20, 30, 40, 50",
]

print("[....] Testing ...\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{'=' * 50}")
    print(f"Test {i}: {prompt}")
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = model.generate(
        **tokenizer(text=text, return_tensors="pt").to("cuda"),
        max_new_tokens=256, temperature=0.7, top_p=0.8, top_k=20, use_cache=True,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the generated part
    if prompt in response:
        response = response.split(prompt)[-1]
    print(response[:300])
    print()

In [ ]:
# ============================================================================
# Save & Export
# ============================================================================
model.save_pretrained("qwen3_4b_distill_lora")
tokenizer.save_pretrained("qwen3_4b_distill_lora")
print("[OK] LoRA adapter saved")

if not SMOKE_TEST:
    # Qwen3.5 GGUF fix: upstream Qwen metadata declares Qwen3_5ForConditionalGeneration
    # which triggers Unsloth/llama.cpp VLM codepath, but Qwen3.5 is text-only.
    # Rewrite to ForCausalLM so llama.cpp safety gate disables mmproj extraction.
    if hasattr(model.config, 'architectures') and model.config.architectures:
        if model.config.architectures[0] == 'Qwen3_5ForConditionalGeneration':
            model.config.architectures = ['Qwen3_5ForCausalLM']
            print('[FIX] Patched architectures -> Qwen3_5ForCausalLM (disables mmproj)')
    model.save_pretrained_merged("qwen3_4b_distill_merged", tokenizer)
    print("[OK] Merged model saved")
    _pushed_merged = False
    model.push_to_hub_merged(HUB_MODEL_ID, tokenizer, token=os.environ.get("HF_TOKEN", ""))
    print(f"[OK] Pushed to {HUB_MODEL_ID}")
    _pushed_merged = True
    _pushed_gguf = False
    try:
        model.save_pretrained_gguf("qwen3_4b_distill_gguf", tokenizer, quantization_method=["q4_k_m", "q5_k_m", "q8_0"])
        print("[OK] GGUF saved")
        model.push_to_hub_gguf(f"{HUB_MODEL_ID}-GGUF", tokenizer, quantization_method=["q4_k_m", "q5_k_m", "q8_0"], token=os.environ.get("HF_TOKEN", ""))
        print(f"[OK] GGUF pushed")
        _pushed_gguf = True
    except Exception as e:
        print(f"[WARN] GGUF export failed (non-fatal): {e}")
        print("[INFO] Training + merge + Hub push succeeded. GGUF can be done separately.")
else:
    print("[SMOKE] Skipping merge/GGUF/push")

print("\n[OK] Done!")

In [ ]:
# ============================================================================
# Hub Round-Trip Validation (non-smoke only)
# ============================================================================
# Loads the pushed merged repo via transformers AutoModel, runs a small
# generation test, then downloads one quantized GGUF and runs it through
# llama-cli to prove the published artifacts are usable by downstream consumers.
if not SMOKE_TEST and globals().get("_pushed_merged", False):
    import gc, shutil, subprocess, tempfile, traceback
    from pathlib import Path as _Path

    _VAL_PROMPT = "The capital of France is"
    _VAL_MAX_TOKENS = 20
    _LLAMA_CLI = "/home/ermia/.unsloth/llama.cpp/build/bin/llama-cli"

    # Free GPU memory held by the training session before reload
    try:
        del model; del trainer  # noqa: F821
    except Exception:
        pass
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

    print(f"\n[VALIDATE] Round-trip test for {HUB_MODEL_ID}")

    # --- Merged safetensors round-trip ---
    merged_ok = False
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        _tok = AutoTokenizer.from_pretrained(HUB_MODEL_ID, trust_remote_code=True)
        _mdl = AutoModelForCausalLM.from_pretrained(
            HUB_MODEL_ID,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True,
        )
        _inputs = _tok(text=_VAL_PROMPT, return_tensors="pt").to(_mdl.device)
        _out = _mdl.generate(**_inputs, max_new_tokens=_VAL_MAX_TOKENS, do_sample=False)
        _decoded = _tok.decode(_out[0], skip_special_tokens=True)
        if len(_decoded.strip()) > len(_VAL_PROMPT):
            merged_ok = True
            print(f"[VALIDATE] Merged repo inference OK")
            print(f"           prompt: {_VAL_PROMPT!r}")
            print(f"           output: {_decoded!r}")
        else:
            print("[VALIDATE] Merged inference returned empty or prompt-only output")
        del _mdl, _tok
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
    except Exception as _e:
        print(f"[VALIDATE] Merged repo load/inference FAILED: {_e}")
        traceback.print_exc()

    # --- GGUF round-trip (only if GGUF push succeeded) ---
    gguf_ok = False
    if globals().get("_pushed_gguf", False):
        try:
            from huggingface_hub import hf_hub_download, list_repo_files
            _gguf_repo = f"{HUB_MODEL_ID}-GGUF"
            _files = list_repo_files(_gguf_repo)
            # Prefer q4_k_m for speed; fall back to first .gguf
            _ggufs = [f for f in _files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()]
            _chosen = next((f for f in _ggufs if "q4_k_m" in f.lower()), _ggufs[0] if _ggufs else None)
            if _chosen is None:
                print(f"[VALIDATE] No .gguf files found on {_gguf_repo}")
            elif not _Path(_LLAMA_CLI).exists():
                print(f"[VALIDATE] llama-cli not found at {_LLAMA_CLI} — skipping GGUF inference")
            else:
                print(f"[VALIDATE] Downloading {_chosen} from {_gguf_repo} ...")
                _local = hf_hub_download(_gguf_repo, _chosen, token=os.environ.get("HF_TOKEN", None))
                print(f"[VALIDATE] Running llama-cli on {_Path(_local).name} ...")
                _cmd = [
                    _LLAMA_CLI, "-m", _local,
                    "-p", _VAL_PROMPT,
                    "-n", str(_VAL_MAX_TOKENS),
                    "--no-display-prompt",
                    "--no-warmup",
                    "-no-cnv",
                ]
                _r = subprocess.run(_cmd, capture_output=True, text=True, timeout=300)
                if _r.returncode == 0 and len(_r.stdout.strip()) > 0:
                    gguf_ok = True
                    print(f"[VALIDATE] GGUF inference OK")
                    print(f"           llama-cli stdout (first 200c): {_r.stdout[:200]!r}")
                else:
                    print(f"[VALIDATE] GGUF inference FAILED rc={_r.returncode}")
                    print(f"           stderr (last 300c): {_r.stderr[-300:]!r}")
        except Exception as _e:
            print(f"[VALIDATE] GGUF round-trip FAILED: {_e}")
            traceback.print_exc()

    print()
    print("=" * 60)
    print(f"[VALIDATE] SUMMARY for {HUB_MODEL_ID}")
    print(f"           merged repo : {'PASS' if merged_ok else 'FAIL'}")
    print(f"           GGUF repo   : {'PASS' if gguf_ok else ('FAIL' if globals().get('_pushed_gguf', False) else 'SKIP (no GGUF pushed)')}")
    print("=" * 60)
elif SMOKE_TEST:
    print("[SMOKE] Skipping Hub round-trip validation")
else:
    print("[VALIDATE] Skipped — no Hub push occurred (training or merge likely failed)")
